In [1]:
# Make relevant library imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data
# Note: The dataset may have encoding issues, so we use 'latin-1'
df = pd.read_csv('AviationData.csv', encoding='latin-1', low_memory=False)

# Inspect the data
print("Data Shape:", df.shape)
print("\nData Types and Non-Null Counts:")
df.info()

print("\nSummary Statistics:")
print(df.describe())

print("\nMissing Values (NaNs) per column:")
print(df.isnull().sum())

Data Shape: (88889, 31)

Data Types and Non-Null Counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50249 non-null  object 
 9   Airport.Name            52790 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87572 non-

In [2]:
# 1. Filter for Airplanes only
df = df[df['Aircraft.Category'] == 'Airplane']

# 2. Filter for Professional builds
df = df[df['Amateur.Built'] == 'No']

# 3. Filter for events from 1983 onwards
# First, convert Event.Date to datetime format
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')
df = df[df['Event.Date'] >= '1983-01-01']

# Check the new shape after filtering
print("Data Shape after filtering:", df.shape)

Data Shape after filtering: (21447, 31)


In [3]:
print(df.columns.tolist())

['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date']


In [4]:
# 1. Clean and create 'Total.Passengers' column
# Fill NaNs with 0 to avoid calculation errors
cols_to_fill = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']
for col in cols_to_fill:
    df[col] = df[col].fillna(0)

df['Total.Passengers'] = df[cols_to_fill].sum(axis=1)

# 2. Create 'Fatal_Serious_Fraction' metric
# Handle cases where Total.Passengers is 0 (avoid division by zero)
# We set these to 0 as no one was onboard to be injured.
df['Fatal_Serious_Fraction'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Passengers'].replace(0, np.nan)
df['Fatal_Serious_Fraction'] = df['Fatal_Serious_Fraction'].fillna(0)

# 3. Clean 'Aircraft.Damage' and create 'Destroyed' flag
print("\nAircraft damage Categories before cleaning:")
print(df['Aircraft.damage'].value_counts(dropna=False))

# Fill missing values with 'Unknown'
df['Aircraft.damage'] = df['Aircraft.damage'].fillna('Unknown')

# Create binary column: 1 if Destroyed, 0 otherwise
df['Is_Destroyed'] = df['Aircraft.damage'].apply(lambda x: 1 if x == 'Destroyed' else 0)


Aircraft damage Categories before cleaning:
Substantial    16990
Destroyed       2316
NaN             1227
Minor            817
Unknown           97
Name: Aircraft.damage, dtype: int64


In [5]:
# Clean up 'Make' (make everything uppercase and strip whitespace)
df['Make'] = df['Make'].str.upper().str.strip()

# Get counts of each Make
make_counts = df['Make'].value_counts()

# Get a list of Makes that have 50 or more occurrences
valid_makes = make_counts[make_counts >= 50].index

# Filter the DataFrame to only keep those makes
df = df[df['Make'].isin(valid_makes)]
print("Data Shape after filtering Makes:", df.shape)

Data Shape after filtering Makes: (17892, 34)


In [6]:
# Check if a Model belongs to only one Make
model_make_check = df.groupby('Model')['Make'].nunique()
ambiguous_models = model_make_check[model_make_check > 1].index
print(f"Number of Models that belong to more than one Make: {len(ambiguous_models)}")

# Create a unique identifier column: 'Make_Model'
df['Make_Model'] = df['Make'] + ' ' + df['Model']

Number of Models that belong to more than one Make: 115


In [7]:
# Standardize string columns (Uppercase and Strip whitespace)
cat_cols = ['Engine.Type', 'Weather.Condition', 'Purpose.of.flight', 'Broad.phase.of.flight']
for col in cat_cols:
    df[col] = df[col].astype(str).str.upper().str.strip()
    # Replace common missing value placeholders with NaN
    df[col] = df[col].replace(['NAN', 'UNKNOWN', 'UNK'], np.nan)

# Clean Number.of.Engines (convert to numeric, set impossible values to NaN)
df['Number.of.Engines'] = pd.to_numeric(df['Number.of.Engines'], errors='coerce')
df.loc[df['Number.of.Engines'] > 8, 'Number.of.Engines'] = np.nan # Assuming no planes have >8 engines

print("\nCleaned 'Purpose.of.flight' top categories:")
print(df['Purpose.of.flight'].value_counts().head())


Cleaned 'Purpose.of.flight' top categories:
PERSONAL              9845
INSTRUCTIONAL         2410
AERIAL APPLICATION     726
BUSINESS               410
POSITIONING            270
Name: Purpose.of.flight, dtype: int64


In [8]:
# Strip whitespace from column names
df.columns = df.columns.str.strip()

# Check if the columns are now recognized
print("Columns found:")
print(df.columns.tolist())
print("\nNon-null counts for first few columns:")
print(df.count().head())

Columns found:
['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date', 'Total.Passengers', 'Fatal_Serious_Fraction', 'Is_Destroyed', 'Make_Model']

Non-null counts for first few columns:
Event.Id              17892
Investigation.Type    17892
Accident.Number       17892
Event.Date            17892
Location              17888
dtype: int64


In [9]:
# Set the threshold
threshold = 10000

# Identify columns that have less than the threshold of non-null values
cols_to_drop = df.columns[df.count() < threshold]

print(f"Dropping {len(cols_to_drop)} columns: {list(cols_to_drop)}")

# Drop the columns
df = df.drop(columns=cols_to_drop)
print("Data Shape after column removal:", df.shape)

Dropping 3 columns: ['Schedule', 'Air.carrier', 'Broad.phase.of.flight']
Data Shape after column removal: (17892, 32)


In [10]:
# Save the cleaned DataFrame to a new CSV file
df.to_csv('AviationData_Cleaned.csv', index=False)
print("Cleaned data saved to 'AviationData_Cleaned.csv'")

Cleaned data saved to 'AviationData_Cleaned.csv'
